In [1]:
import pickle
import json
import os
import glob

# Đường dẫn
JSON_PATH = r'D:\Big_project_2025\Bao_Cao_TTTN\products_final.json'
IMAGE_DIR = r"D:\Big_project_2025\zTai_Lieu\DATA_THTN\moho_data\images"
TEXT_PKL = "moho_text_embeddings.pkl"
IMAGE_PKL = "moho_all_images_embeddings.pkl"

print("📊 --- BÁO CÁO KIỂM CHỨNG KHO LƯU TRỮ VECTOR --- 📊\n")

# 1. Kiểm tra Text Vectors
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    raw_products = json.load(f)
with open(TEXT_PKL, 'rb') as f:
    text_vectors = pickle.load(f)

num_sp_json = len(raw_products)
num_sp_vectors = len(text_vectors)

print(f"[TEXT] Số lượng sản phẩm trong JSON: {num_sp_json}")
print(f"[TEXT] Số lượng vector sản phẩm đã trích xuất: {num_sp_vectors}")
print(f"==> Tình trạng: {'✅ KHỚP' if num_sp_json == num_sp_vectors else '❌ LỆCH'}")
print("-" * 50)

# 2. Kiểm tra Image Vectors
# Đếm số ảnh thực tế trong thư mục (jpg, jpeg, png, webp)
actual_images = []
for ext in ['*.jpg', '*.jpeg', '*.png', '*.webp']:
    actual_images.extend(glob.glob(os.path.join(IMAGE_DIR, "**", ext), recursive=True))

with open(IMAGE_PKL, 'rb') as f:
    image_vectors = pickle.load(f)

num_img_folder = len(actual_images)
num_img_vectors = len(image_vectors)

print(f"[IMAGE] Tổng số file ảnh trong thư mục: {num_img_folder}")
print(f"[IMAGE] Tổng số vector ảnh đã trích xuất: {num_img_vectors}")
print(f"==> Tình trạng: {'✅ KHỚP' if num_img_folder == num_img_vectors else '❌ LỆCH'}")
print("-" * 50)

# 3. Thông tin kỹ thuật cho báo cáo
sample_key = list(text_vectors.keys())[0]
vector_dim = text_vectors[sample_key]['name_vector'].shape[0]
print(f" [INFO] Kích thước vector (Dimensions): {vector_dim}")
print(f" [INFO] Định dạng lưu trữ: Pickle (Python Serialized)")

📊 --- BÁO CÁO KIỂM CHỨNG KHO LƯU TRỮ VECTOR --- 📊

[TEXT] Số lượng sản phẩm trong JSON: 193
[TEXT] Số lượng vector sản phẩm đã trích xuất: 193
==> Tình trạng: ✅ KHỚP
--------------------------------------------------
[IMAGE] Tổng số file ảnh trong thư mục: 954
[IMAGE] Tổng số vector ảnh đã trích xuất: 954
==> Tình trạng: ✅ KHỚP
--------------------------------------------------
 [INFO] Kích thước vector (Dimensions): 512
 [INFO] Định dạng lưu trữ: Pickle (Python Serialized)


In [2]:
import pickle
import numpy as np

# --- XEM FILE TEXT EMBEDDINGS ---
print("--- [SOI] FILE: moho_text_embeddings.pkl ---")
with open("moho_text_embeddings.pkl", "rb") as f:
    text_data = pickle.load(f)

# Lấy thử 1 ID sản phẩm đầu tiên để xem
sample_id = list(text_data.keys())[0]
sample_content = text_data[sample_id]

print(f"Sản phẩm mẫu: {sample_id}")
print(f"- Tên SP: {sample_content['metadata']['name']}")
print(f"- Kích thước vector Name: {sample_content['name_vector'].shape}") # Sẽ là (512,)
print(f"- 5 giá trị đầu tiên của vector Name: {sample_content['name_vector'][:5]}") 
print("-" * 50)


# --- XEM FILE IMAGE EMBEDDINGS ---
print("\n--- [SOI] FILE: moho_all_images_embeddings.pkl ---")
with open("moho_all_images_embeddings.pkl", "rb") as f:
    image_data = pickle.load(f)

# Lấy thử 1 đường dẫn ảnh đầu tiên để xem
sample_img_path = list(image_data.keys())[0]
sample_img_vector = image_data[sample_img_path]

print(f"Ảnh mẫu: {sample_img_path}")
print(f"- Kích thước vector ảnh: {sample_img_vector.shape}") # Sẽ là (512,)
print(f"- 5 giá trị đầu tiên của vector ảnh: {sample_img_vector[:5]}")

--- [SOI] FILE: moho_text_embeddings.pkl ---
Sản phẩm mẫu: ban-an-oslo-901
- Tên SP: Bàn Ăn Gỗ Cao Su MOHO OSLO 901
- Kích thước vector Name: (512,)
- 5 giá trị đầu tiên của vector Name: [-0.05629741  0.00370236 -0.01931885 -0.10196123 -0.06048415]
--------------------------------------------------

--- [SOI] FILE: moho_all_images_embeddings.pkl ---
Ảnh mẫu: main\ban-an-go-cao-su-moho-oslo-901_1.jpg
- Kích thước vector ảnh: (512,)
- 5 giá trị đầu tiên của vector ảnh: [-0.6667717  -0.13160163  0.09486106 -0.06308141  0.04916489]


In [ ]:
import pickle
import torch
import json
import os
from sentence_transformers import util

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN
# ==========================================
JSON_PATH = r'D:\Big_project_2025\CODE_THTN\products_final.json'
TEXT_PKL = "moho_text_embeddings.pkl"
IMAGE_PKL = "moho_all_images_embeddings.pkl"

device = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 1. LOAD DỮ LIỆU
# ==========================================
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    products_data = json.load(f)

with open(TEXT_PKL, 'rb') as f:
    text_embeddings = pickle.load(f)

with open(IMAGE_PKL, 'rb') as f:
    image_embeddings = pickle.load(f)

# ==========================================
# 2. CHỌN SẢN PHẨM MẪU ĐỂ KIỂM ĐỊNH
# ==========================================
# Chọn sản phẩm đầu tiên: Bàn Ăn Gỗ Cao Su MOHO OSLO 901
sample = products_data[0]
p_id = sample['product_id']
p_name = sample['content']['name']
p_main_img = sample['images']['main']

print(f"🧪 Đang kiểm định chất lượng Vector cho sản phẩm: {p_name}")

# Lấy Vector Văn bản (Sử dụng Name Vector)
vec_text = torch.tensor(text_embeddings[p_id]['name_vector']).to(device)

# Lấy Vector Ảnh (Tìm trong file image pkl)
# Lưu ý: Key trong file image_pkl là đường dẫn tương đối, ví dụ: 'main\ten-anh.jpg'
vec_image = None
search_key = os.path.join("main", p_main_img)

# Chuẩn hóa đường dẫn để tìm kiếm chính xác
for key in image_embeddings.keys():
    if os.path.normpath(key) == os.path.normpath(search_key):
        vec_image = torch.tensor(image_embeddings[key]).to(device)
        break

# ==========================================
# 3. TÍNH ĐỘ TƯƠNG ĐỒNG COSINE
# ==========================================
if vec_image is not None:
    # Tính Cosine Similarity
    similarity_score = util.cos_sim(vec_text, vec_image).item()
    
    print("\n" + "="*40)
    print("📊 KẾT QUẢ KIỂM ĐỊNH")
    print("="*40)
    print(f"Sản phẩm: {p_name}")
    print(f"ID:      {p_id}")
    print(f"Ảnh:     {p_main_img}")
    print(f"Độ tương đồng Cosine: {similarity_score:.4f}")
    print("="*40)
    
    if similarity_score > 0.2:
        print("✅ Kết luận: Vector Ảnh và Chữ khớp nhau về ngữ nghĩa.")
    else:
        print("⚠️ Kết luận: Độ tương đồng thấp, cần kiểm tra lại model.")
else:
    print(f"❌ Không tìm thấy vector ảnh cho key: {search_key}")

🧪 Đang kiểm định chất lượng Vector cho sản phẩm: Bàn Ăn Gỗ Cao Su MOHO OSLO 901

📊 KẾT QUẢ KIỂM ĐỊNH
Sản phẩm: Bàn Ăn Gỗ Cao Su MOHO OSLO 901
ID:      ban-an-oslo-901
Ảnh:     ban-an-go-cao-su-moho-oslo-901_1.jpg
Độ tương đồng Cosine: 0.2106
✅ Kết luận: Vector Ảnh và Chữ khớp nhau về ngữ nghĩa.
